In [23]:
V = 1 + (ord('Г') + ord('Д')) % 10
print("Вариант: "+ str(V))

Вариант: 8


# 1. Подготовка данных для построения классификационных моделей.

## Описание набора данных

In [24]:
import pandas as pd
import numpy as np

data = pd.read_csv("tweet_emotions.csv")
data.head()

print("\nРазмерность данных:", data.shape)

print("\nКоличество уникальных классов:", data['sentiment'].nunique())
print("Список классов:", data['sentiment'].unique())


Размерность данных: (40000, 3)

Количество уникальных классов: 13
Список классов: ['empty' 'sadness' 'enthusiasm' 'neutral' 'worry' 'surprise' 'love' 'fun'
 'hate' 'happiness' 'boredom' 'relief' 'anger']


## Векторизация и предобработка данных

In [31]:
from collections import defaultdict
import re

no_registry_account = data['content']
lower_registry = data['content'].str.lower()



stop_words_removed = lower_registry

def get_specific_words_docs(df, class_col='sentiment', text_col='content', min_doc_ratio=0.01, max_other_ratio=0.2):
    classes = data[class_col].unique()
    word_doc_counts = defaultdict(lambda: defaultdict(int))
    class_doc_counts = defaultdict(int)

    for c in classes:
        texts = data[data[class_col]==c][text_col].str.lower()
        class_doc_counts[c] = len(texts)
        for doc in texts:
            words = set(doc.split())
            for w in words:
                word_doc_counts[w][c] += 1

    specific_words = set()
    for w, counts in word_doc_counts.items():
        for c in classes:
            if counts[c] / class_doc_counts[c] >= min_doc_ratio:
                other_classes = [oc for oc in classes if oc != c]
                if all(counts[oc] / class_doc_counts[oc] < max_other_ratio for oc in other_classes):
                    specific_words.add(w)
    return specific_words

specific_words_account = get_specific_words_docs(data)

print(f"Количество специфичных слов: {len(specific_words_account)}")
print("Примеры специфичных слов:", list(specific_words_account)[:20])



Количество специфичных слов: 423
Примеры специфичных слов: ['ugh', 'amazing', 'live', "won't", 'me', 'good', '2', 'full', 'woke', 'done', 'friday', 'all.', 'had', 'enjoy', 'new', 'having', 'man', 'work', 'much', 'stop']


# 2. Построение классификаторов и анализ качества работы при разных способах предобработки.

## Оценки качества моделей

In [26]:
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import MultinomialNB
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.metrics import accuracy_score, f1_score

y = data['sentiment']
results = []

dataset_variants = {
    "Без учета регистра": no_registry_account,
    "Приведенные к нижнему регистру": lower_registry,
    "С удалением стоп-слов": stop_words_removed,
    "Выделение только специфичных слов": specific_words_account
}

models = {
    'NaiveBayes': MultinomialNB(),
    'RandomForest': RandomForestClassifier(n_estimators=100, max_depth=15, random_state=94),
    'LogisticRegression': LogisticRegression(max_iter=1000),
    'SVM': LinearSVC()
}

for variant_name, dataset_variant in dataset_variants.items():
    print(f"\n=== Набор данных: {variant_name} ===")
    vectorizer = CountVectorizer()
    if variant_name=="Выделение только специфичных слов":
      vectorizer = CountVectorizer(vocabulary = specific_words_account)
      X = vectorizer.fit_transform(lower_registry)
      X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=94, stratify=y)
      feature_names = np.array(vectorizer.get_feature_names_out())
    elif variant_name=="С удалением стоп-слов":
      vectorizer = CountVectorizer(stop_words='english')
      X = vectorizer.fit_transform(lower_registry)
      X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=94, stratify=y)
      feature_names = np.array(vectorizer.get_feature_names_out())
    else:
      vectorizer = CountVectorizer()
      X = vectorizer.fit_transform(dataset_variant)
      X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=94, stratify=y)
      feature_names = np.array(vectorizer.get_feature_names_out())

    for model_name, model in models.items():
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)
        acc = accuracy_score(y_test, y_pred)
        f1 = f1_score(y_test, y_pred, average='weighted')
        print(f"{model_name:18s}  Accuracy={acc:.4f}  F1-score={f1:.4f}")

        results.append({
            "Вариант предобработки": variant_name,
            "Модель": model_name,
            "Accuracy": round(acc, 4),
            "F1-score": round(f1, 4),
            "Кол-во признаков": X.shape[1]
        })

df_results = pd.DataFrame(results)
df_results_sorted = df_results.sort_values(['Accuracy', 'F1-score'], ascending=[False, False])
print("\n=== Сводная таблица результатов ===")
display(df_results_sorted)


=== Набор данных: Без учета регистра ===
NaiveBayes          Accuracy=0.3061  F1-score=0.2475
RandomForest        Accuracy=0.2772  F1-score=0.1679
LogisticRegression  Accuracy=0.3409  F1-score=0.3137


/usr/local/lib/python3.12/dist-packages/sklearn/svm/_base.py:1249: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(


SVM                 Accuracy=0.2951  F1-score=0.2823

=== Набор данных: Приведенные к нижнему регистру ===
NaiveBayes          Accuracy=0.3061  F1-score=0.2475
RandomForest        Accuracy=0.2772  F1-score=0.1679
LogisticRegression  Accuracy=0.3409  F1-score=0.3137


/usr/local/lib/python3.12/dist-packages/sklearn/svm/_base.py:1249: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(


SVM                 Accuracy=0.2951  F1-score=0.2823

=== Набор данных: С удалением стоп-слов ===
NaiveBayes          Accuracy=0.3083  F1-score=0.2584
RandomForest        Accuracy=0.2727  F1-score=0.1643
LogisticRegression  Accuracy=0.3351  F1-score=0.3072


/usr/local/lib/python3.12/dist-packages/sklearn/svm/_base.py:1249: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(


SVM                 Accuracy=0.2923  F1-score=0.2787

=== Набор данных: Выделение только специфичных слов ===
NaiveBayes          Accuracy=0.3247  F1-score=0.2963
RandomForest        Accuracy=0.3116  F1-score=0.2529
LogisticRegression  Accuracy=0.3376  F1-score=0.3022
SVM                 Accuracy=0.3407  F1-score=0.3015

=== Сводная таблица результатов ===


,Вариант предобработки,Модель,Accuracy,F1-score,Кол-во признаков
2,Без учета регистра,LogisticRegression,0.3409,0.3137,48212
6,Приведенные к нижнему регистру,LogisticRegression,0.3409,0.3137,48212
15,Выделение только специфичных слов,SVM,0.3407,0.3015,423
14,Выделение только специфичных слов,LogisticRegression,0.3376,0.3022,423
10,С удалением стоп-слов,LogisticRegression,0.3351,0.3072,47917
12,Выделение только специфичных слов,NaiveBayes,0.3247,0.2963,423
13,Выделение только специфичных слов,RandomForest,0.3116,0.2529,423
8,С удалением стоп-слов,NaiveBayes,0.3083,0.2584,47917
0,Без учета регистра,NaiveBayes,0.3061,0.2475,48212
4,Приведенные к нижнему регистру,NaiveBayes,0.3061,0.2475,48212


In [27]:
y = data['sentiment']
dataset_variants = {
    "Без учета регистра": no_registry_account,
    "Приведенные к нижнему регистру": lower_registry,
    "С удалением стоп-слов": stop_words_removed,
    "Выделение только специфичных слов": specific_words_account
}

models = {
    'NaiveBayes': MultinomialNB(),
    'RandomForest': RandomForestClassifier(n_estimators=100, max_depth=15, random_state=94),
    'LogisticRegression': LogisticRegression(max_iter=1000),
    'SVM': LinearSVC()
}

for variant_name, dataset_variant in dataset_variants.items():
    print(f"\n=== Набор данных: {variant_name} ===")
    vectorizer = CountVectorizer(max_features=5000)
    if variant_name=="Выделение только специфичных слов":
      vectorizer = CountVectorizer(max_features=5000, vocabulary = specific_words_account)
      X = vectorizer.fit_transform(lower_registry)
      X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=94, stratify=y)
      feature_names = np.array(vectorizer.get_feature_names_out())
    else:
      vectorizer = CountVectorizer(max_features=5000)
      X = vectorizer.fit_transform(dataset_variant)
      X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=94, stratify=y)
      feature_names = np.array(vectorizer.get_feature_names_out())

    for model_name, model in models.items():
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)
        importances = None
        if model_name == 'RandomForest':
            importances = model.feature_importances_
        elif model_name in ['LogisticRegression', 'SVM']:
            importances = np.abs(model.coef_).mean(axis=0)
        elif model_name == 'NaiveBayes':
            importances = np.abs(model.feature_log_prob_).mean(axis=0)

        if importances is not None:
            top_indices = np.argsort(importances)[::-1][:10]
            top_features = feature_names[top_indices]
            top_values = importances[top_indices]

            print(f"{model_name:18s}Топ-10 признаков по важности:")
            for feat, val in zip(top_features, top_values):
                print(f"    {feat:25s} {val:.6f}")



=== Набор данных: Без учета регистра ===
NaiveBayes        Топ-10 признаков по важности:
    sean                      9.963313
    aaaah                     9.909994
    gluten                    9.909994
    cig                       9.909994
    amen                      9.909994
    tmr                       9.909994
    apartments                9.909994
    suspended                 9.878804
    archie                    9.878804
    davis                     9.878804
RandomForest      Топ-10 признаков по важности:
    love                      0.050683
    happy                     0.039536
    day                       0.022678
    sad                       0.022638
    mothers                   0.014461
    my                        0.013333
    miss                      0.013035
    great                     0.012796
    mother                    0.010945
    thanks                    0.010611
LogisticRegressionТоп-10 признаков по важности:
    hate                      0.79

# 3. Поиск слов, которые уникальны для класса. Оценка участия таких показателей в моделях.

In [34]:
from collections import defaultdict
import numpy as np

texts = lower_registry
tokenized = [t.split() for t in texts]
class_words = defaultdict(set)
for text, label in zip(tokenized, y):
    class_words[label].update(text)

unique_words = {}
all_words = set().union(*class_words.values())

for label, words_set in class_words.items():
    other_words = all_words - words_set
    uniq = words_set - (all_words - words_set)
    unique_words[label] = uniq

print("Количество уникальных слов по классам:")
for label, words in unique_words.items():
    print(f"{label}: {len(words)} слов")

for label, words in unique_words.items():
    print(f"\nПримеры уникальных слов для класса '{label}': {list(words)[:10]}")

vectorizer = CountVectorizer()
X_vec = vectorizer.fit_transform(texts)
vocab = set(vectorizer.get_feature_names_out())

found_counts = {label: len(words & vocab) for label, words in unique_words.items()}

table_data = []
for label in unique_words:
    n_unique = len(unique_words[label])
    n_used = found_counts[label]
    ratio = n_used / n_unique if n_used > 0 else 0
    table_data.append([label, n_unique, n_used, ratio])

df = pd.DataFrame(table_data, columns=['Класс', 'Уникальных слов', 'Использовано в модели', 'Отношение'])
print(df)


Количество уникальных слов по классам:
empty: 3690 слов
sadness: 14946 слов
enthusiasm: 3664 слов
neutral: 22862 слов
worry: 22536 слов
surprise: 8717 слов
love: 11944 слов
fun: 7768 слов
hate: 5851 слов
happiness: 16117 слов
boredom: 1170 слов
relief: 6085 слов
anger: 846 слов

Примеры уникальных слов для класса 'empty': ['nice.', 'fml', 'awesome.', 'lame', 'pain?', 'networking', '(vid)', 'wrk', 'delish', '@iantalbot']

Примеры уникальных слов для класса 'sadness': ['@yurges', 'tree-lined', 'fml', 'repairs...', '@sheistheemily', 'guy.....unfortunatly', 'to.', 'pendant', 'edge', 'good']

Примеры уникальных слов для класса 'enthusiasm': ['nice.', 'kids.', 'fml', 'awesome.', 'advert', 'lame', 'www.crossroads.net/gocincinnati', 'ways.', 'lol,', "shower.we're"]

Примеры уникальных слов для класса 'neutral': ['@maroonedmarla', 'live360', '@alandavies1', 'to.', 'aww,', '@msdahlia', 'thang!', 'matinee(sp?', '@sophielovemcfly', '@thenewnicole']

Примеры уникальных слов для класса 'worry': ['hi